# Laboratorio 1.4 · Procesamiento en un solo nodo

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** tres años de ventas de la tienda, **10 millones de líneas de pedido**

Una sola pregunta de negocio, cinco formas de responderla:

> **Ingreso neto mensual (en MXN) y unidades por categoría y ciudad, solo 2025 y solo canal web.**

| Parte | Qué haces | Lo que vas a descubrir |
|---|---|---|
| 1 | Generar el historial | Cuánto pesa lo mismo en CSV y en Parquet |
| 2 | La misma pregunta en pandas, Polars y DuckDB | Tiempo y memoria pico de cada motor |
| 3 | El plan de Polars | Cómo el optimizador lee solo lo necesario |
| 4 | Parquet por dentro | Row groups, estadísticas min/max y particiones |
| 5 | Más grande que la memoria | Por partes, streaming y fuera de memoria |
| 6 | Tipos de datos | La misma columna puede ocupar 10 veces más |
| 7 | Arrow | Pasar datos entre motores sin copiar |

## Preparación

In [ ]:
%%capture
!pip -q install -U "polars>=1.10" "duckdb>=1.1" pyarrow

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode("UEsDBBQAAAAIAPazRV326cQ9ZgAAAGwAAAAZAAAAcHJvY2VzYW1pZW50by9fX2luaXRfXy5weQ3MQQqDQAwF0L2n+GRloQgFVy6LB/AKcSbgYEzKZFz0XD2CF6v7xyOipXqS4KOINYcYTkO4OsyzT/iwZY4nFleugS/mM+3z+yZrFYhiK9G8FlZkgTLaHWVGHxLl+hlew/gYiKj7A1BLAwQUAAAACAACtEVdx6ISzXkHAABEEgAAIgAAAHByb2Nlc2FtaWVudG8vZ2VuZXJhcl9oaXN0b3JpYWwucHmFWNtu4zYavvdTEOqNPOsoPiRpGkAFMknmsJh008lMUSAwBEaibe5KpJaUkkmLPkwve9GLoo+QF+v3UwdLPkwMxJbI/3z4+DOe572TttBG8pQlgj0IVXBLTylnhRQq4YwzYWOe8jOWyTTVSlT7z38qUZHmIpGJHjEjcqOTMpb3qbDBYPDZapZqsJ4xlj8VK63YQcZAEwvLMwgvdLAUShhuolVrxcHBQqYQPBlXHyxYnkoYkvBC2zXlgHU+fgzpp+ztazLo4/k10ywTStsRKy1vZc4qkcPBlY2NvBdnkJFK8iOI7QPb9ykVszrV7OL2J7Y0XEEF9GXkHYMXRiy5ef6TOzpYJzLOUqwleriWn3Pz/1IUO+WLFKG1kAcnCqEQTIhttd5UnK2oSJtEKEjfJbPZY7k2bCHiFWc+yDgya0gPQlloipHRj/BFl7ldGxmBspCx1CThkCupw3N8DjNhw+vrw1cdhV1K1EchYeg7+SAgK04ptaJ1ecSqskDymqWB53kDmcHGgnGzxKoVzbtdlZDWvBUyE4OF0agbXqxSec/qjRu8DhoqVWb5E0OCVd4s5TqFVFrL08Hg9ur6/YcP5yxk351+e3J8NLh4//ny/PLqFit33sXl9c/eiHlvS56g0P+LeNHrNWXDGPFELzeluE/d8o+lMM9/FNxoR/T8h0Fx0uMHmbnf13qpi+ff6fGWq0LypaO8KG2svfng4vzT1dv/fHx/Xim/SkVcmOe/lYwd9zuNaqKHf5dLhEpYev6oc7d5Kci5au21SFPxS6353mi3eM2hBC3sBHCToEudbh1LxZ3yH84/1G4/inva43lOP1W3g2QwSMSCeh4dIk2vJvxkMULxIN0K/R7rtMwUt8Mz14rI6c2eqjhrmKqimo6nx66oJtNDbuKVfGhrmQqDhBXm6axt8GQRPBpZiKgm8lsTnHEFFEb3T2Frj2MUX2KRF+zTUy6ujNEGIMS+YTd1WSAty1KfUQeveCwyban5AFMctI+t5qaYquUACMQt6p9wz67Ns7V59TaiFBQ6ciz+sBOwhTYZL0Kv8bVjv1TL1v7+crRI+YM2oYcwCW9Yp6fGTb+CRogmfAsn4wjwRn8jpqKmEcPjdqntxHBaLVlBoM7DukHqVNaAG7ouq3VUUZWLejMQXwB11h+u01R1bmAyIKLocdUs2f8SaQiOYJYNP5lSVNtGLaFL5QFhq84COMjLtIiw7tcG1toVVRdoCRaCXJhFFOuSutRHYIii8Rk0eRpcIiFvDM+E/2trpVeTRDLxzkgpul0thT/phoz9i+E9KVA8IUikKk6OhqOODFkmnPhhYhCvtIyF30BKVw5SGd4FMwQ6mBzR1wxf41P6OqGvY1qj3fG39EUk49m8q8mKZUYnZV/XnadK8eBgZWFEXJIyenmQuTffNoB0zaZOdSP8tyqibUF8JWANzc6IrQW8FDJeiCUd3BtRa9GwJ63LmcNDqaMU9cYrA3BuqcQnKaleKuqq1D8OEMxx8N2GGDZdO+x+v2Fv6Fi0dMTOGH/+q2797Pl328w/2FL6QYrs3oiDBDXnnpj/GoFmb6RiT2g4dHWDNYnkFEAfEQQGiKiKT/3mA+6QdgRnAnM6i7B3gqTMaNmbJMif4Ethqsao0yOsrlrjcSWM8ElRkBRBhqNp5Q8JZtzZ5w/Z9yGbQMc0QDlNgnFrGLL2BSI6IU+FcoKGNWxQjThFh05fYMvMr/Wj+hBpVx0kAEkVsND60HF6Eh0RgjgR/WLapN5VKD22dX6pDBvCu37m5x1v71pVB2wyrxJbzTBfqWNHsKOIq9nwhfKthtyK21+zV26ww0PUEiQMA25JhL9LhBvHiN1lseNMnaV5w+xRhRDEnRzd2bk3hOQmES0JbSciLXhNM9zFXNLOaA/0baeoj36Va6N9ONC89Vocx+oaF7vCTxp5/QDNpjsavVQS05XU3V7v1cgrJx5kdKD6rucnwdpi1hOK09fh4yaCjqlRel8OPybueXI83xEAdJ1I+CYSX//8A0Hv59tLb97pqLvg1KH78byPgoqnmwjoRrIeK3HO+uwNftX3iWriwK2lPmjRvN76KuMNt2mb4WmLfj2OYGCJ3KUgsvIXgbnBDQ7DGjevtq8p29cNjFQEoEgTZk/MPTHeMqkOM/6Frh0xJuz13QPXJQPIZe6GQlcTbqpbXtd6iwnMr5tn+IIvW3ejF5zaPeU2YZPFKqqGMuu7wRXIgve1MUDiJ4EpDDVH4+yO7RqoOyWw/dlyomsMHLjzSDnVGJR482Fv0NkbkM1rmLdx4O9l3Lqt1ZxGFKVR7FfP1SlquK5Xr8EmKmvXrDumM+B0Nby5U7AXDA/1GmX3Lfeegg5QUoVPPy6P2J6+ejUdbwqrbf66wNaxfUJ/w5iNUTeKFM6QKGJhyLwoyrhUUeRVE2+Ok6a5uwbnZlnSmHZDb4buJ6irnCb40HvrRnW6fK9e+FdLk6KAJwmuD5VI36v/e4ECcKcTYJPuFG5E7kz8+3gr5701i7fxf5Raqzt6A+cO8WOyr+sF+vzmusGD5sLBgwoeh4N/AFBLAwQUAAAACADos0VdPvy8fKkJAADoHAAAGAAAAHByb2Nlc2FtaWVudG8vbW90b3Jlcy5weaVZyW7kyBG98ysSHMAiBzUcddtjAwLqoJaqxzK0WYsXCAUiRWaVsodkcrioVS3oY+boQ598MOCj9WN+uXEvaSwXuiEyl8hYX0QGXdc9piTlZUpJXrB1nVWUxIxkbC0iLkjBypolGItERmJeVjyrRElSUQlMyZV1RkqRCJKJWASOc26I7BGerbEEE6wSJGVZWdOEeCwjJ3879ckGG3lMYxDJRUEiWuHE4vkrxUzEa8zMNN33u+9/wJh6jmgGGp/ZbeBciZiWJMH/VZ2B0wyEChbxWxyQgF1a5EyJkpA7sC0Kjp0bvN9DnnssAt+HtKIfC5oyKUdOM0lRyok9UiPCYeXPNUslS6KIWTbDogJ6EjEDxyLNC3FLC4I1pIL0YKerGgpCDzSqQD+rmGRKqxmTec3KigaO67rOqhAp6FZ3Cb8lPIUyKnKOV8cxL3Ed/RTf2jfDJv7lcTMmElroscRxro7Oz8KD/ZMPR2fh9eUhmZN3fwh2nYOz4+uT0/1LvN+4KSvdGXGN1jlVL0rr8slaRj4bK4bpQ+YundPFweJy/+LIkFmx6E7vTbgUMuRqP/QCpithXmG1ihvSubSRCHFCRQsu5BAOimrsVi8pDBlTs4smONJxYrYi4Yrj1YtX/p5D8ItXYCBe3ViplkEk8o3nm0kt4VKv0c8BLatNzjxIVP3+d267spHWLm8Htu7pacVs6401O1eJoGpvUIg6i733LZFW+5ZEZ8QSKKuis0ObqFluXkdrC1bVBeJ1FZTwj/CeJnA470WzL8EgK1kV8ixmD15ciHx+VdTMhwG+Id8NftYNh+PaVnoyRCiVNbDDS3jGaDkjsTGe9Y8ScuQxzqUx9hSIo8qLyfetA5WBGTaqN262dZ+dH2zT52OTfrgxDFn/XQZxFWwYInk+V3Djk9+QZo3xQznnAnlcfzlFM0hZsWZew/jNzTgKGtsuZ0Rk894C3xCwEmB/P6asmczezqSWcQWkAY42MtpIWgYpzb1HF6Dr7pF3wS5oARXwPMCJp0ldwbVKvs48eM58Qms4pLqbqY1bfp2omHdUqgFhSb5t+B0ig5zz3pHvmgUtTix9zGmBjb+3/K4RZvnt5hVnnwEstafPP9KkZH5A12vPBv7c62FWWaeuP+tJ4vWi3a7pBZ9BrMI3CGaDorz3THIy0YAkcIwE+fyLTD7p87+RNlQKy4vnX6KKR0ilCUPCQVY6uPyLyjuJTKnIJyyVgqm0hpzy/A8kQ55UBS0CmVkUakAzMps0Z/ZYHAaqDSrJpAooo1S8uyr5lSyMoc9y3niBL8O6L6INyZGYJzKpkuj5nzFfC9BjhYBoG5bFAkyqTcQzqT5RuTip0wwPGYtYCa+gpf9/StaDCyOdBYuZObCct2nOijcBgec6626BQDU5AYEzSPZlY3Si7AosS4ISDmeZI3ylFsHiJZOTXc77ADrrQqKkthU/Z53pLTBpdGd47QV1IP0K2z1wAyV5xvq+RU7P72GnXaWx02+x0+9T/SR41mJmULKERZW3HTn9CeCcoNjIZwlOQqm/BUkbWp95dRcal1AAOCG8AkDPfwkCu7jRUYyGF4lkdnCIgL5FQLugRcCBIvs/RfHzHcsaO5hcoA0h0d8PKjk/SAF+IDBcfOYl85AphtZS0BoCW1+C1sGeHqyOpQ+Am94AW+2qLsKahX6DNDq8GF3DKUc4YyITMJqiLAfxlMWcVmIP14IYwEpxK0FksU8sqnFHwMUgFbJIFxIBgFAaPAxApYhffscBSRKZbgtG9otCfP6VOKSTwAgOBkE9CUd+gxY2R8HjNWteq4hE4oamLpcOlPEKSI656uLQa1zZ8rBjkAkWxvaI+YoVPIY58oKnUse0wL0I+U1KA+KCiLziKf9C7cVPGidDDpAgJudtQhCNGfoK36oW6VoKEya1GaKCZjSF5ykKRoKq2Ow1Xm3OkdMNLZatoaS52+w2UcAeIpZX5Aq1+QI+U+xh7Btyz4pS31cRBlCEvUhrLb14UHPAlO6byekawxgB2ClVJ1OuKSjk9RvWlTmkPcDfw4VeqxloRCqWsUJewG3xIW2pq4//LRBaBW/1vuniA0YIuwWIUsDAF0c2ncjah7hNH34YZ+3LPx+DfynL5eJ4cXBFNLIngQJ7n+DGC9QDJgQN6gEtAtOqsFa7vD7BFotvapdFvy1rZAUcDJC/qXyDBvFlLjjYv1yQv/5xcYoJjejgeAe4vUOu5OhjFT2RxTEWvSOL00N1egdEnY8XZyfkUWv1iSTOn86OTkkPhnYe46fvR8XDjk9ycn15dPoj8Tqp199GYFheYH9k97fZ1ncgysWCqPKhUbMuIsj+6SFROgI2Q0bUDjvOjxdn1+fkw9/J/vExObs4XFyYF9VG0aGgeyXbK1BjfetzJgps7Rlz2MH2a2QjCE6xR0qeyXbSmsoAoKrWoq87fyALYVHyB28qDjSfQflz4uGMYCUAgJVXRfNBRoaDz6WLK5vNV+5Y2X143vFRGgTxqoU0oxEEUZgAUisaC8vjzEoxd3+7u3vywR3pSee75uKhmmbJ81eQUS0ze/8AePAVjzhNtOpu6Sch9SYhJGHIrzDkFxxXl1T2ECPx69UmD50bIQCDWaaw204F7EGmcOat3MvFleZno+WUbvNoGHzacbdtqViah9ru2Ko2Qamh0VqV5jvulP0kobcZT6Jb13B47xtte7OlA9STlw5757mrs596F82ZvBYqxRahIjB/F+7u7sr/rcnNIfY+NkoOTfGEF1k96ZumTNCwMEBmjWM2spZSGpKjSBa3eM5UWxT7pIMAFV7OF2/pDb3cbHlj52h7B0bzaeWRjVA9tLK6AuqSl6/SdcmQ07sXzS3Xa+QYZU7+hc0HRvTbokSfOtd/bzz958Xullmypbk1ojlsb73ewHrhctW0qgwTb+xUjZns9KrGKni9VTW4qjUaantVZmxrq8rMv9Cp6rlOQPMcwaaN8dbO1c1Nr1nV60IvzbWpaZLBKwFdIOw1TPhvbpkp2lMtL3h+hizq2XbKY8vhXufrwpPCu5Ozq7OLhfye8KihyADRf/4lAUhuaDtn7my4xKTuzrKmm6OXmrIXS5troFrcuUCOl9obSmelrDLtQpMd+8f3Sw8sfXI+Xi8u9sPDRXiyODlDkG+RsamqXxO2D8jd1Q3kj2VpKu5GnRPXhbFkNuOjMiAoDVoBu5WEklLXGbe0ZON661SQOxoxkiF1yIqguGf6G5o0RaG+iZkaQn5EE1ENgcj5prrDyRvVA0w4ShD5XVDnv/L5q7wR0nJ48YNnN5/zGsezX4d8579QSwMEFAAAAAgA1LRFXXCOzhTDBgAA7Q8AABYAAABwcm9jZXNhbWllbnRvL21lZGlyLnB5lVfbbuQ0GL7PU5hwk0AmdIvgYkSQKrYrrdSl1W5XXAyjyJN4pi6JHWyn24L2kbjiEfbF+H7HOUy7RRBV08T+z4fvt+M4fiNqWclPfyt221vH16ziNWetdtqwShsjmFCsV6wzuhJWM95x4wTDL6jAa5jtmZOi7TR7oPdWtNpIzq5e/3TJrFRRpZXjrVTcMN4wI6zTLGn4RPh7L5YKGrkTkK00mCEVBljBanHXi+ZOkAQrrRMtZ7oDnZN3Os2ji1kcqFtZE4vFL9/3oMqY3llh7riqNRPNpMv72ulKW/A00opDDwrLkn2vKqkVJ+d/kVj7YLOo5dXlOzh5IVV/D6XRezJXV7xZM9Y9uBut2KoNwuGxUE7nQ4xWq4qbTjjOau60LW/gA1nbRHEcRxLBM45xc0BYrRi/b61W47vtd4NcO608TK8IR7eXzcToZCuivdGwhbsbBJSFjSt8RiNVh2hwy/DX1dOa7Z1sooE591Ug7Mj96v3527Py5Xn55vzN5dvXZxl7c3l9+fb8XRRdXZz9fP364uKMFcwsXCIXMjI1Wxq1DNAoPOiKpEI5aogh+hwp3peV7pUTJkkjEPSN4zXtHxBN50wSGL2WHCG827zYpsn0cbpNoymvn5XKVmxQOovPnS6RClSmmyV9u81AWIv74hVvrEijzkjlEnIxr/u2s8mf8agpXjMD+XUyLmTsNP2Ypj7dUVSL/dA8Y6FlLNRHuo4YHpD95JsveJf/EAh/TEbCR32penGnc/ZybJShJbOpLTpZUYMK34HBSzKGtH2Q7mYqovyaGA03Dy+lERXUPyBGqBLXdoNx9FjeSPRO4UsqwVbKvmHxLDpEL54YDJd/jORlSYrKMs3BoGFtkhIDyiH8A/2XA8QQOMDmqfzRvqrqQWL8esdJi5i0dFAxt0p+BYxQyYZSKO5F1Tu+a0TG4lUVZ2yq2YxNWbAoKG/iGOXZwnTYHRxPt9mk87OPdbXuXbG05fXVOYmohTFP1524d8W16WFd9aEuKFrp7NXgejYksQhdml8NEpIu72QN804mhg83iC/Dum4aJE9a9rNWYu3DGpCwRTssQBhuAjTREMDNht3IW818fxpKvOE7fsuPHHbmYf0kAsG8lt8n9JqNhudey0Mp1V4j1cba9IhX3FeiG8EnPzdGm6fCd0bw345toF62jRBdcpKfnKaPirNErDMmSBq9UtzySrdtj2bnDhmd6OUeW0a43qhKY2h8UbCTYwOGTXbU4BRS1BJ5Wra7ecFrxOekOUfdyA6O266RrpFK2CTdrF5sN+sX359sP2aec9LX+omMWVJ4+MwbzWubzD49Ky59ImIzWbclYPaI5JP0DTv96qvTk5kjODgyInU1IsLrGQaHug/ghThSe5qxT7IRqIohCEZg1UvC0osZ1K7RgHzkxuzm7Gt2J4zcIyXDKQThp553mvC6ocEchlDNFXV8K22rP4NhI1kxveEAM84nXz/cwrCSCDzyxrQQz8DriYBL3JL5ewEcqiQck4eeN17wZpuFHP/5caCGCqVbVOYEIZgQE2ZLQCpSs16k5TbYeJRwElMSo+HqIJJl7NLjMmwzthyBz4yQIxa5n9QGGKDAJO1mLuUt7Sjthl2cCtjx7g+jhOVq+rRDZ/faZWvNFi/UrB8bOYd8gqsnChY0xSz2WNKQrc2QFqr6ndZN0mAMTAwpKwo2rIzy0n9H8+GhyMxSNpu4FRYFFBOaHAhF/Yfsa17TG3AGbyDZbknhrOz/cKY5Jwgffv+zkQsrY6kONMPK9l4hl6ulHcd7ULJDuSLdQNPvgspZp2+NnHeYqDVOOr7IgXJj/S+R8TPl8ty4jMMAmhF0UbFL+BqLVzR0wH+6v/INvlh5XiMhSDlFCGpD0eQ4UiaDP+kCx4MyvxsWcZTz0p2Hs4LA8iV3/JXhrUh8oIDOwpX+uJiEWA0s8M5zbYKobS6t4mOG55ofRfv/eW10l1S66Vtli9EITxuA25MBnCG+LBXMKEsqurgsWy5VWcaDZDofjdeM/Mwc+hYz/oq+TIKCqzBYHLCkwOmzHa54I/r6OxqOQjhpAprxthtOYNNNJtjT5byuSx5EJ/F080FhY3JwBL2IH12CnmX1l7dVLVahSiCDV4OBxCpKhxMTFm9E0xVxGCsMN0xDjSXpakMLXp0fK4rufDucnIUajz5BuU9j7gNDJtjQbcMZfxp3PJ8G3uPbEGWW597ishbleLDy9RrmUEr3Cpre6jB21iA//lXNfeCBfXForxcXVgTGj0XvjK563OHY1XDzfMBxztLt2Qjz6S9yHThvP/1FSeMWXv4DUEsDBBQAAAAIAEi0RV14AxFK5gQAABoLAAAgAAAAcHJvY2VzYW1pZW50by9wYXJxdWV0X2ludGVybm8ucHmNVttu4zYQffdXDAiklgxH3vWifTDWLdLYQQs0F+SCoggCgZFom61Eakkq69TwJ/Qj+gF96GtfCjQ/1uFFspxtsjVyMcnR4cyZOTMihJxyRRXkTBgl8R/UAi6o+lAzMwElP8JSybrSQ2Da0PzpD214RjWUXIxKuoZHyOqn34WRoBlUNUMATQuDiLSABS8MgieEkB4vK6kMGF6y3kLJEnJqmF1BOGnW/rSiZlXw++bwApe9BiOvs1/y+2ZVUZGjQ/hT5e3eI1Xoe1L5QNzhh16vl7OFjSn1MUWqNnQImSzqUtApWbBsRUk86QF+0OcbQW0IFBBzRwXaC9A1iKe/SuYps0YaqWBFywvu4oMBGqJlTVVOc6mBCWtWcYYmBVCVrfiDjB1F9tqSGQpT9DYJSTjhBXOOxok9Q5aoM+T5Gu3sVqKzFStpYmTqok79OoqTJTPpgrMiT7nI2ToK7sQOwDs9hds7v8QYOXABioolixywqMt0R1cgxn7Usrm7PY543B5jqeC5Wib+wgh9jRMsH8Nt9ejWzrmQ0KpiIo82pMUiE+BDIO4YvyNQ8ATLkCDDuIdXJPjNrum6WdP1sMXe/5DyPs1kWSle8lxaTFnjnYhspKFFev9oWKr5rwxGMB4Mxm+G8Dbe+ogUM7USWF3JDMk/UbRkkXMt/qSiUsEypqnisimunOmcDWFF9aul9s/fx15GuqM5uDifXT79dnRmS84wwVRIGtbQbRc4/gbmBfqpvQydAEFjLgvGVCF1W10uby8JwEdrZEaFy94t0nPr6L2Dr6c+khi+AL+NWbiD9+H+uy5RBRORQ4mH7rtaNkQZzlCfqWsLMlrU2HPYc4oUq7ANZFwKpqfvdgxdu2d9g7LCErq2UfoWk3uNOladPmfYImbfApbxz9I2N3jApCBzmVSKY7vYCU5/KDDaBS6v5j/Mj6/h+Pzm7DoaxHB0BQUXjGLRXeLeLLq6OY0wLIMAOQygUgzdTGvBjc04Rjt2Dz1gWPRZHZ5cnp/Cxke8hR+/m1/Og7fI7PX3p/Or66PTC+hvHBmTg58OD8rDg3zbh6OzWbB8v2fo6OoY7hoIRmxpdPzkEoM7Qy6H7m+r9HSn9C7hHYlzYffwaduTk4qpBQqoRvdVtBN69xbflRPkM8LfOFkwk60QtGNu/hsODsNtraFPGxoDX4QF1y4CbJ5Y4Vh9UQjU7InUyzocvYs7PIQStE2A4kxKrUhQpRlVGH4j1GkzhKLxm/GX2APGtg+E2tw7/AoP7FlgLEdn7Yxq8OKE6rSSmq9D+D71tt9u2ihJaPFOqlLhCKYKW9OCKEbzNMyuCGtiO/KF2MyzfkyGn6J4BMxFK4VXwdLG/kVUXxUNIn36U45Kpl8HrTqPjQaD0aBFRxb5A/MGBg24sM3IqJo1925fmktClvcKS9iTaEs30Jlww0oddcrWDCFqVOuEGLua+1zfifdmks0S8WYYbXM70WyJ1SUtA3gN8dfgormPuAtxw/3f7qS0ANLlhSCksVF45KSQH60MJns9I7wZuN5k+d30n6Wtb3H7nVWD5yUSzPvbJgFkD343qFBCdgTuDYbuHAuOvMRXw9kt6TxOUzt5cEA473dgW9uKN+6+7c4f6+/kfwMSLQvpXq681LA/6OnbMdnLYPtWYRefG+OJxvck/4bUZD3u/QtQSwECFAMUAAAACAD2s0Vd9unEPWYAAABsAAAAGQAAAAAAAAAAAAAApIEAAAAAcHJvY2VzYW1pZW50by9fX2luaXRfXy5weVBLAQIUAxQAAAAIAAK0RV3HohLNeQcAAEQSAAAiAAAAAAAAAAAAAACkgZ0AAABwcm9jZXNhbWllbnRvL2dlbmVyYXJfaGlzdG9yaWFsLnB5UEsBAhQDFAAAAAgA6LNFXT78vHypCQAA6BwAABgAAAAAAAAAAAAAAKSBVggAAHByb2Nlc2FtaWVudG8vbW90b3Jlcy5weVBLAQIUAxQAAAAIANS0RV1wjs4UwwYAAO0PAAAWAAAAAAAAAAAAAACkgTUSAABwcm9jZXNhbWllbnRvL21lZGlyLnB5UEsBAhQDFAAAAAgASLRFXXgDEUrmBAAAGgsAACAAAAAAAAAAAAAAAKSBLBkAAHByb2Nlc2FtaWVudG8vcGFycXVldF9pbnRlcm5vLnB5UEsFBgAAAAAFAAUAbwEAAFAeAAAAAA=="))).extractall(".")
print("Paquete procesamiento instalado")

In [ ]:
import inspect, os, time
import duckdb, pandas as pd, polars as pl, pyarrow as pa
from IPython.display import Code, display
from procesamiento import generar_historial, motores, parquet_interno
from procesamiento.medir import comparar

def ver(funcion):
    display(Code(inspect.getsource(funcion), language="python"))

print("pandas", pd.__version__, "| Polars", pl.__version__, "| DuckDB", duckdb.__version__, "| CPU:", os.cpu_count())
!free -h | head -2

## Parte 1 · El historial de ventas

Tarda alrededor de un minuto. Genera los mismos 10 millones de filas en tres formatos: un CSV, un Parquet
y un Parquet particionado por año y mes (además de una copia ordenada por fecha que usaremos en la parte 4).

In [ ]:
CARPETA = "datos_historial"
print(generar_historial.generar(CARPETA, filas=10_000_000))
!du -sh datos_historial/*

In [ ]:
pl.read_parquet(f"{CARPETA}/lineas.parquet", n_rows=5)

El mismo contenido: **709 MB en CSV contra unos 170 MB en Parquet**. Parquet comprime por columna y guarda los tipos.

## Parte 2 · Una pregunta, cinco motores

Primero mira cómo se escribe la misma consulta en cada motor:

In [ ]:
ver(motores._pandas_consulta)

In [ ]:
ver(motores._polars_consulta)
print(motores.SQL)

Ahora la carrera. Cada motor corre en **su propio proceso** para medir tiempo y **memoria pico** sin que uno contamine al otro
(ábrelo con `ver(medir)` si tienes curiosidad). Tarda alrededor de un minuto.

In [ ]:
tabla = comparar(CARPETA, motores.MOTORES)
tabla

Lee la tabla con calma:

- **`mismo_resultado`** debe ser `True` en todos. Un motor rápido que da otra respuesta no sirve.
- **pandas sobre CSV** es el caso más común en la práctica, y el más caro: lee las 10 columnas como texto, interpreta fechas y luego filtra.
- **pandas sobre Parquet** ya mejora mucho solo por cambiar el formato y leer las columnas necesarias.
- **Polars diferido y DuckDB** no leen lo que no necesitan: filtran y proyectan **dentro** de la lectura.

## Parte 3 · El plan de Polars

En modo diferido, Polars primero arma un plan y lo optimiza. Compara el plan **como lo escribimos** contra el **optimizado**:

In [ ]:
plan = motores.plan_polars_lazy(CARPETA)
print(plan.explain(optimized=False)[-900:])

In [ ]:
print(plan.explain()[-1300:])

Busca dentro de `Parquet SCAN [lineas.parquet]`:

- **`PROJECT 8/10 COLUMNS`**: *projection pushdown*, solo lee las columnas que usa la consulta.
- **`SELECTION: ...`**: *predicate pushdown*, el filtro de 2025 y canal web se aplica **mientras lee**, no después.

DuckDB hace lo mismo con su propio optimizador:

In [ ]:
d = CARPETA
print(duckdb.sql("EXPLAIN " + motores.SQL.format(tc=17.0, d=d, lineas=f"read_parquet('{d}/lineas.parquet')")).fetchall()[0][1][:1800])

## Parte 4 · Parquet por dentro

Un archivo Parquet se divide en **row groups** (bloques de filas). En el pie del archivo se guardan el mínimo y el máximo
de cada columna en cada row group. Si un filtro cae fuera de ese rango, el motor **ni siquiera lee** ese bloque.

In [ ]:
print("Sin ordenar:")
display(parquet_interno.row_groups(f"{CARPETA}/lineas.parquet").head(4))
print("Ordenado por fecha:")
display(parquet_interno.row_groups(f"{CARPETA}/lineas_ordenado.parquet").head(4))

¿Ves la diferencia? Sin ordenar, **cada** row group va de enero de 2023 a diciembre de 2025: ninguno se puede saltar. Ahora midamos una consulta de diciembre de 2025:

In [ ]:
parquet_interno.comparar_saltos(CARPETA)

Las tres dan el mismo resultado, pero el archivo ordenado lee **1 de 20** row groups y el particionado solo abre la carpeta `mes=12`.
Ordenar y particionar por la columna con la que más filtras es de las optimizaciones más baratas que existen.

## Parte 5 · Más grande que la memoria

Imagina que el CSV no cabe en RAM. Comparamos cuatro estrategias, otra vez cada una en su proceso:

- **pandas completo**: la referencia (carga todo).
- **pandas por partes**: `chunksize`, agregados parciales que se combinan al final.
- **Polars streaming**: el motor procesa el CSV por lotes.
- **DuckDB con 300 MB**: le ponemos un límite de memoria artificialmente bajo; si no le alcanza, usa disco.

In [ ]:
comparar(CARPETA, motores.FUERA_DE_MEMORIA)

In [ ]:
ver(motores.pandas_chunks_csv)

## Parte 6 · Los tipos de datos importan

La misma columna `canal` (solo tres valores distintos) con tres representaciones distintas en pandas:

In [ ]:
canal = pl.read_parquet(f"{CARPETA}/lineas.parquet", columns=["canal"]).to_series().to_pandas()
variantes = {
    "object (texto de Python)": canal.astype(object),
    "string de Arrow": canal.astype("string[pyarrow]"),
    "category": canal.astype("category"),
}
pd.DataFrame({"MB": {k: round(v.memory_usage(deep=True) / 2**20, 1) for k, v in variantes.items()}})

Una columna de texto de baja cardinalidad guardada como objetos de Python puede ocupar decenas de veces más que como `category`.
En pandas, revisar los tipos (`df.dtypes`, `df.memory_usage(deep=True)`) es lo primero cuando algo no cabe.

## Parte 7 · Arrow: el idioma común

pandas (con tipos Arrow), Polars y DuckDB entienden el mismo formato columnar en memoria: **Apache Arrow**.
Pasar datos de uno a otro casi no cuesta.

In [ ]:
lineas_pl = pl.read_parquet(f"{CARPETA}/lineas.parquet", columns=["canal", "cantidad", "precio_unitario"])

inicio = time.perf_counter()
por_canal = duckdb.sql("SELECT canal, SUM(cantidad * precio_unitario) AS venta FROM lineas_pl GROUP BY canal").pl()
print(f"DuckDB consultó un DataFrame de Polars de {len(lineas_pl):,} filas en {time.perf_counter() - inicio:.2f} s")
por_canal

DuckDB encontró la variable `lineas_pl` por su nombre y la leyó directamente a través de Arrow, sin convertirla.
Con `.pl()`, `.df()` o `.arrow()` el resultado vuelve como Polars, pandas o Arrow.

## Retos

### Reto 1 · Tu propia pregunta
Escribe «top 10 productos por ingreso en CDMX en 2025» en **dos** motores y verifica que coinciden.

In [ ]:
# Pista: en DuckDB
# duckdb.sql(f"SELECT producto_id, SUM(...) AS ingreso FROM read_parquet('{CARPETA}/lineas.parquet') l "
#            f"JOIN read_parquet('{CARPETA}/clientes.parquet') c USING (cliente_id) WHERE ... ORDER BY ingreso DESC LIMIT 10").df()

### Reto 2 · Compresión
¿Cuánto cambian el tamaño y el tiempo de escritura con distintos códecs?

In [ ]:
muestra = pl.read_parquet(f"{CARPETA}/lineas.parquet", n_rows=2_000_000)
filas = []
for codec in ["uncompressed", "snappy", "zstd", "gzip"]:
    inicio = time.perf_counter()
    muestra.write_parquet(f"prueba_{codec}.parquet", compression=codec)
    filas.append({"códec": codec, "MB": round(os.path.getsize(f"prueba_{codec}.parquet") / 2**20, 1),
                  "escritura_s": round(time.perf_counter() - inicio, 2)})
    os.remove(f"prueba_{codec}.parquet")
pd.DataFrame(filas).set_index("códec")

### Reto 3 · El problema de los archivos pequeños
El mismo millón de filas en 1 archivo o en 2 000 archivos. ¿Cuánto tarda en leerlo DuckDB?

In [ ]:
import shutil
millon = pl.read_parquet(f"{CARPETA}/lineas.parquet", n_rows=1_000_000)
shutil.rmtree("pequenos", ignore_errors=True); os.makedirs("pequenos")
millon.write_parquet("uno.parquet")
for i, parte in enumerate(millon.iter_slices(500)):
    parte.write_parquet(f"pequenos/parte_{i:05d}.parquet")
for fuente in ["uno.parquet", "pequenos/*.parquet"]:
    inicio = time.perf_counter()
    n = duckdb.sql(f"SELECT COUNT(*), SUM(cantidad) FROM read_parquet('{fuente}')").fetchone()
    print(f"{fuente:22s} {time.perf_counter() - inicio:.3f} s  {n}")

Cada archivo tiene su propio encabezado, metadatos y una apertura en disco (o una petición HTTP si está en Cloud Storage o S3).
Miles de archivos pequeños son lentos y caros: por eso se compactan.

## Cierre

| Hoy viste | Principio |
|---|---|
| CSV contra Parquet | El formato cambia más que el motor |
| Projection y predicate pushdown | No leer lo que no necesitas |
| Row groups ordenados y particiones | Saltar datos es más rápido que procesarlos |
| Streaming y fuera de memoria | Que no quepa en RAM no obliga a usar un clúster |
| Tipos de datos | Revisar `dtypes` antes de comprar más memoria |
| Arrow | Elegir la herramienta por paso, sin pagar conversiones |

**Tarea:** la parte 1.4 del **kit local** (`kit_pipeline_1_4.zip`): corre `medir` en tu máquina y agrega tu pregunta a `motores.py`.

**Próxima sesión (2.1):** pipelines ETL en Python, con transformaciones modulares, deduplicación, SCD y pruebas.

Para liberar espacio en Colab: `!rm -rf datos_historial`